# Final two-dataset study
Run from the implementation directory. Uses supplied results and models. See README_REVISION.md for full training. Pooled confidence intervals and equal-subject tests address different estimands.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
from src.wisdm import extract_features, ACTIVITIES
from src.models import predict
print('Final study: UCI HAR and WISDM, subject-disjoint evaluation')

Final study: UCI HAR and WISDM, subject-disjoint evaluation


In [2]:
data = np.load('data/wisdm_prepared.npz')
print('WISDM features:', data['X'].shape)
print('Training windows:', len(data['train']), 'Test windows:', len(data['test']))
assert set(data['subjects'][data['train']]).isdisjoint(data['subjects'][data['test']])
print('Train/test subjects are disjoint.')

WISDM features: (4933, 51)
Training windows: 3125 Test windows: 1808
Train/test subjects are disjoint.


In [3]:
for dataset in ['uci', 'wisdm']:
    metrics = pd.read_csv(f'results_revision/{dataset}/test_metrics.csv')
    summary = metrics[(metrics.kind == 'clean') | ((metrics.kind == 'dropout') & (metrics.severity == .5))]
    print(dataset, '\n', summary.groupby(['model', 'kind']).macro_f1.mean().round(4))

uci 
 model                kind   
Dropout-LR           clean      0.9490
                     dropout    0.8709
Expanded Dropout-LR  clean      0.9490
                     dropout    0.8709
Expanded LR          clean      0.9500
                     dropout    0.8596
LR                   clean      0.9500
                     dropout    0.8596
RBF-SVM              clean      0.9515
                     dropout    0.7923
Random forest        clean      0.9245
                     dropout    0.7513
Name: macro_f1, dtype: float64
wisdm 
 model          kind   
Dropout-LR     clean      0.8628
               dropout    0.6350
LR             clean      0.8458
               dropout    0.4629
LR matched C   clean      0.8593
               dropout    0.4134
RBF-SVM        clean      0.8873
               dropout    0.5181
Random forest  clean      0.8563
               dropout    0.6404
Name: macro_f1, dtype: float64


In [4]:
for dataset in ['uci', 'wisdm']:
    comparisons = pd.read_csv(f'results_revision/{dataset}/paired_comparisons.csv')
    print(dataset, comparisons[(comparisons.reference == 'LR') & (comparisons.kind == 'dropout') & (comparisons.severity == .5)].to_string(index=False))

uci     method reference    kind  severity    delta      low     high
Dropout-LR        LR dropout       0.5 0.011272 0.007787 0.014887
wisdm     method reference    kind  severity    delta      low     high
Dropout-LR        LR dropout       0.5 0.172114 0.144039 0.189278


In [5]:
model = joblib.load('results_revision/wisdm/models/dropout-lr.joblib')
features = extract_features(data['windows'][data['test'][:5]])
labels = predict(model, features)
print('Predicted activities:', [ACTIVITIES[int(label)-1] for label in labels])
print('True activities:', [ACTIVITIES[int(label)-1] for label in data['y'][data['test'][:5]]])

Predicted activities: ['Jogging', 'Jogging', 'Walking', 'Walking', 'Downstairs']
True activities: ['Jogging', 'Jogging', 'Walking', 'Walking', 'Upstairs']
